# Auction Bot Detector
**End-to-End ML Pipeline — Stage 4: Model Training**

This notebook covers two approaches to detect auction bots:
1. **Transformer Classifier** — Supervised learning with sequence classification
2. **Autoencoder** — Unsupervised anomaly detection

### Dataset Summary
- 1,984 labelled bidders (103 bots, 1,881 humans)
- 8 features per bid: `time`, `time_diff`, `device`, `country`, `auction`, `merchandise`, `same_auction_flag`, `same_device_flag`
- Sequences truncated/padded to 500 bids per bidder


## Setup

In [ ]:
# Install dependencies
!pip install imbalanced-learn -q

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt

print('Libraries loaded successfully.')

In [ ]:
# Mount Google Drive and load data
from google.colab import drive
drive.mount('/content/drive')

X = np.load('/content/drive/MyDrive/auction-bot-detector/data/processed/X_sequences.npy')
y = np.load('/content/drive/MyDrive/auction-bot-detector/data/processed/y_labels.npy')

print(f'X shape: {X.shape}')
print(f'Total bots:   {int(y.sum())}')
print(f'Total humans: {int(len(y) - y.sum())}')

In [ ]:
# Split FIRST, then scale (prevents data leakage)
n_samples, n_steps, n_features = X.shape

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Fit scaler on train only — never on test
scaler = StandardScaler()
X_train_reshaped = X_train.reshape(-1, n_features)
X_test_reshaped  = X_test.reshape(-1, n_features)

X_train_scaled = scaler.fit_transform(X_train_reshaped)
X_test_scaled  = scaler.transform(X_test_reshaped)

X_train = X_train_scaled.reshape(len(X_train), n_steps, n_features)
X_test  = X_test_scaled.reshape(len(X_test), n_steps, n_features)

print(f'Train size: {len(X_train)} | Test size: {len(X_test)}')
print(f'Train bots: {int(y_train.sum())} | Test bots: {int(y_test.sum())}')
print(f'X_train mean: {X_train.mean():.4f} | X_train std: {X_train.std():.4f}')

---
## Approach 1: Transformer Classifier

**Architecture:**
- Embedding layer: maps 8 features → 32 dimensional space
- Positional Encoding: preserves bid order information  
- Transformer Encoder: attention mechanism connects each bid to every other bid
- Classification Head: outputs Bot/Human probability

**Handling class imbalance:**
- SMOTE oversampling on train set only
- Focal Loss to focus learning on hard (rare bot) examples


In [ ]:
# SMOTE oversampling on TRAIN set only
# Test set stays clean (no synthetic data)
X_train_2d = X_train.reshape(len(X_train), n_steps * n_features)

print(f'Before SMOTE: Bots={int(y_train.sum())} | Humans={int(len(y_train)-y_train.sum())}')

smote = SMOTE(random_state=42, k_neighbors=5)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_2d, y_train)

# Reshape back to 3D
X_train_smote = X_train_resampled.reshape(-1, n_steps, n_features)
y_train_smote = y_train_resampled

print(f'After SMOTE:  Bots={int(y_train_smote.sum())} | Humans={int(len(y_train_smote)-y_train_smote.sum())}')
print(f'Test set unchanged: Bots={int(y_test.sum())} | Humans={int(len(y_test)-y_test.sum())}')

In [ ]:
# Convert to PyTorch tensors
X_train_tensor = torch.FloatTensor(X_train_smote)
X_test_tensor  = torch.FloatTensor(X_test)
y_train_tensor = torch.FloatTensor(y_train_smote)
y_test_tensor  = torch.FloatTensor(y_test)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset  = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)

print(f'Train batches: {len(train_loader)} | Test batches: {len(test_loader)}')

In [ ]:
# Positional Encoding
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() *
                            (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

In [ ]:
# Transformer Model
class BotDetectorTransformer(nn.Module):
    def __init__(self, input_dim=8, d_model=32,
                 nhead=4, num_layers=1, dropout=0.3):
        super().__init__()

        # Step 1: Embedding — converts 8 features to d_model dimensions
        self.embedding = nn.Linear(input_dim, d_model)

        # Step 2: Positional Encoding — tells model bid order
        self.pos_encoding = PositionalEncoding(d_model)

        # Step 3: Attention — connects each bid to every other bid
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        # Step 4: Classification Head — Bot or Human decision
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 16),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.embedding(x)       # 8 features -> d_model
        x = self.pos_encoding(x)    # add position info
        x = self.transformer(x)     # attention across all bids
        x = x.mean(dim=1)           # average across 500 bids
        x = self.classifier(x)      # Bot/Human probability
        return x.squeeze()

In [ ]:
# Focal Loss — focuses learning on hard examples (rare bots)
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, predictions, targets):
        bce_loss = nn.BCELoss(reduction='none')(predictions, targets)
        pt = torch.exp(-bce_loss)
        focal_loss = self.alpha * (1-pt)**self.gamma * bce_loss
        return focal_loss.mean()

# Setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

transformer_model = BotDetectorTransformer().to(device)
criterion = FocalLoss(alpha=0.75, gamma=2)
optimizer = torch.optim.Adam(
    transformer_model.parameters(),
    lr=0.001,
    weight_decay=0.01  # regularisation to prevent overfitting
)
scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer, step_size=10, gamma=0.5
)

print(f'Total parameters: {sum(p.numel() for p in transformer_model.parameters())}')

In [ ]:
# Training Loop with overfitting check
def train_transformer(model, train_loader, test_loader,
                      criterion, optimizer, scheduler,
                      device, epochs=50):

    train_losses = []
    test_losses  = []

    for epoch in range(epochs):

        # Training phase
        model.train()
        total_train_loss = 0
        correct = 0
        total = 0

        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()
            predictions = model(X_batch)
            loss = criterion(predictions, y_batch)
            loss.backward()
            optimizer.step()

            total_train_loss += loss.item()
            predicted = (predictions > 0.5).float()
            correct += (predicted == y_batch).sum().item()
            total += len(y_batch)

        # Evaluation phase (no gradient updates)
        model.eval()
        total_test_loss = 0

        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)
                predictions = model(X_batch)
                loss = criterion(predictions, y_batch)
                total_test_loss += loss.item()

        avg_train_loss = total_train_loss / len(train_loader)
        avg_test_loss  = total_test_loss  / len(test_loader)
        accuracy = correct / total * 100

        train_losses.append(avg_train_loss)
        test_losses.append(avg_test_loss)
        scheduler.step()

        if (epoch + 1) % 10 == 0:
            print(f'Epoch {epoch+1}/{epochs} | '
                  f'Train Loss: {avg_train_loss:.4f} | '
                  f'Test Loss: {avg_test_loss:.4f} | '
                  f'Accuracy: {accuracy:.2f}% | '
                  f'LR: {scheduler.get_last_lr()[0]:.6f}')

    return train_losses, test_losses

t_train_losses, t_test_losses = train_transformer(
    transformer_model, train_loader, test_loader,
    criterion, optimizer, scheduler, device
)

In [ ]:
# Overfitting check — plot train vs test loss
epochs_range = range(1, 51)
plt.figure(figsize=(10, 5))
plt.plot(epochs_range, t_train_losses, label='Train Loss')
plt.plot(epochs_range, t_test_losses,  label='Test Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Transformer: Train vs Test Loss')
plt.legend()
plt.grid(True)
plt.show()

gap = abs(t_train_losses[-1] - t_test_losses[-1])
print(f'Final Train Loss: {t_train_losses[-1]:.4f}')
print(f'Final Test Loss:  {t_test_losses[-1]:.4f}')
print(f'Gap: {gap:.4f}')
if gap < 0.05:
    print('Status: No overfitting ✅')
elif gap < 0.15:
    print('Status: Slight overfitting ⚠️')
else:
    print('Status: Overfitting detected ❌')

In [ ]:
# Threshold analysis for Transformer
transformer_model.eval()
t_preds_prob = []
t_labels = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        preds = transformer_model(X_batch)
        t_preds_prob.extend(preds.cpu().numpy())
        t_labels.extend(y_batch.numpy())

t_preds_prob = np.array(t_preds_prob)
t_labels = np.array(t_labels)

print('Transformer Threshold Analysis:')
print(f'{"Threshold":<12} {"Precision":<12} {"Recall":<12} {"F1":<12}')
print('-' * 48)

for threshold in [0.3, 0.4, 0.5, 0.6, 0.7]:
    preds = (t_preds_prob > threshold).astype(float)
    p = precision_score(t_labels, preds, zero_division=0)
    r = recall_score(t_labels, preds, zero_division=0)
    f = f1_score(t_labels, preds, zero_division=0)
    print(f'{threshold:<12} {p:<12.3f} {r:<12.3f} {f:<12.3f}')

In [ ]:
# Final Transformer evaluation at best threshold
best_threshold = 0.4
t_final_preds = (t_preds_prob > best_threshold).astype(float)

print(f'Transformer Final Results (threshold={best_threshold}):')
print(classification_report(t_labels, t_final_preds,
      target_names=['Human', 'Bot']))

---
## Approach 2: Autoencoder (Anomaly Detection)

**Key idea:** Train only on human sequences. Model learns what 'normal' bidding looks like.
At detection time, bots reconstruct poorly (high error) because their patterns are abnormal.

**Architecture:**
- Encoder: compresses each bid (8 features → 8)
- Bottleneck: compresses entire sequence (500×8 → 32)
- Decoder: reconstructs sequence from bottleneck
- Detection: high reconstruction error = bot


In [ ]:
# Prepare data for Autoencoder — train on HUMANS ONLY
X_train_humans = X_train[y_train == 0]

print(f'Training autoencoder on humans only:')
print(f'Human sequences: {len(X_train_humans)}')
print(f'Bot sequences:   0 (intentionally excluded)')

X_ae_train_tensor = torch.FloatTensor(X_train_humans)
X_ae_test_tensor  = torch.FloatTensor(X_test)

# Input and target are the SAME (reconstruction task)
ae_train_dataset = TensorDataset(X_ae_train_tensor, X_ae_train_tensor)
ae_train_loader  = DataLoader(ae_train_dataset, batch_size=32, shuffle=True)

print(f'Train batches: {len(ae_train_loader)}')

In [ ]:
# Autoencoder Model
class BidAutoencoder(nn.Module):
    def __init__(self, n_features=8, n_steps=500, bottleneck=32):
        super().__init__()

        # Encoder: compress each bid
        self.encoder = nn.Sequential(
            nn.Linear(n_features, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU()
        )

        # Bottleneck: compress entire sequence
        self.bottleneck_down = nn.Linear(n_steps * 8, bottleneck)
        self.bottleneck_up   = nn.Linear(bottleneck, n_steps * 8)

        # Decoder: reconstruct each bid
        self.decoder = nn.Sequential(
            nn.Linear(8, 16),
            nn.ReLU(),
            nn.Linear(16, n_features)
        )

        self.n_steps = n_steps
        self.n_features = n_features

    def forward(self, x):
        batch_size = x.shape[0]
        encoded = self.encoder(x)
        flat = encoded.reshape(batch_size, -1)
        compressed = self.bottleneck_down(flat)
        expanded = self.bottleneck_up(compressed)
        expanded = expanded.reshape(batch_size, self.n_steps, 8)
        reconstructed = self.decoder(expanded)
        return reconstructed


# Masked MSE — only calculate loss on real bids, not padding zeros
def masked_mse(reconstructed, original):
    mask = (original.abs().sum(dim=2) != 0).float()
    mask = mask.unsqueeze(2).expand_as(original)
    diff = (reconstructed - original) ** 2
    masked_diff = diff * mask
    loss = masked_diff.sum() / (mask.sum() + 1e-8)
    return loss


# Setup
ae_model = BidAutoencoder().to(device)
ae_criterion = masked_mse
ae_optimizer = torch.optim.Adam(
    ae_model.parameters(), lr=0.001, weight_decay=0.01
)
ae_scheduler = torch.optim.lr_scheduler.StepLR(
    ae_optimizer, step_size=10, gamma=0.5
)

print(f'Autoencoder parameters: {sum(p.numel() for p in ae_model.parameters())}')

In [ ]:
# Train Autoencoder
def train_autoencoder(model, train_loader, criterion,
                      optimizer, scheduler, device, epochs=50):
    train_losses = []

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for X_batch, _ in train_loader:
            X_batch = X_batch.to(device)
            optimizer.zero_grad()
            reconstructed = model(X_batch)
            loss = criterion(reconstructed, X_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        train_losses.append(avg_loss)
        scheduler.step()

        if (epoch + 1) % 10 == 0:
            print(f'Epoch {epoch+1}/{epochs} | '
                  f'Loss: {avg_loss:.6f} | '
                  f'LR: {scheduler.get_last_lr()[0]:.6f}')

    return train_losses

ae_train_losses = train_autoencoder(
    ae_model, ae_train_loader, ae_criterion,
    ae_optimizer, ae_scheduler, device
)

In [ ]:
# Calculate reconstruction errors for all test sequences
ae_model.eval()
ae_errors = []

with torch.no_grad():
    for i in range(len(X_test)):
        x = torch.FloatTensor(X_test[i:i+1]).to(device)
        reconstructed = ae_model(x)
        error = masked_mse(reconstructed, x).item()
        ae_errors.append(error)

ae_errors = np.array(ae_errors)

# Separate bot and human errors
ae_bot_errors   = ae_errors[y_test == 1]
ae_human_errors = ae_errors[y_test == 0]

print(f'Human reconstruction error: Mean={ae_human_errors.mean():.4f} | Std={ae_human_errors.std():.4f}')
print(f'Bot reconstruction error:   Mean={ae_bot_errors.mean():.4f} | Std={ae_bot_errors.std():.4f}')
print(f'Separation: Bot mean is {ae_bot_errors.mean()/ae_human_errors.mean():.2f}x higher than Human mean')

In [ ]:
# Visualise error distribution
plt.figure(figsize=(10, 5))
plt.hist(ae_human_errors, bins=50, alpha=0.7, label='Human', color='blue')
plt.hist(ae_bot_errors,   bins=50, alpha=0.7, label='Bot',   color='red')
plt.xlabel('Reconstruction Error')
plt.ylabel('Count')
plt.title('Autoencoder: Human vs Bot Reconstruction Error')
plt.legend()
plt.grid(True)
plt.show()

print(f'Overlap: Humans above 0.5: {(ae_human_errors > 0.5).sum()} / {len(ae_human_errors)}')
print(f'Overlap: Bots below 0.5:   {(ae_bot_errors < 0.5).sum()} / {len(ae_bot_errors)}')

In [ ]:
# Threshold analysis for Autoencoder
print('Autoencoder Threshold Analysis:')
print(f'{"Threshold":<12} {"Precision":<12} {"Recall":<12} {"F1":<12}')
print('-' * 48)

for threshold in [0.5, 0.7, 0.9, 1.1, 1.3, 1.5]:
    preds = (ae_errors > threshold).astype(float)
    p = precision_score(y_test, preds, zero_division=0)
    r = recall_score(y_test, preds, zero_division=0)
    f = f1_score(y_test, preds, zero_division=0)
    print(f'{threshold:<12} {p:<12.3f} {r:<12.3f} {f:<12.3f}')

---
## Results Comparison


In [ ]:
# Side by side comparison
print('=' * 60)
print('RESULTS COMPARISON')
print('=' * 60)

print('\n--- Transformer (threshold=0.4) ---')
t_preds_04 = (t_preds_prob > 0.4).astype(float)
print(classification_report(t_labels, t_preds_04,
      target_names=['Human', 'Bot']))

print('\n--- Autoencoder (threshold=0.7) ---')
ae_preds_07 = (ae_errors > 0.7).astype(float)
print(classification_report(y_test, ae_preds_07,
      target_names=['Human', 'Bot']))

print('\n--- Key Limitation ---')
print('Only 103 labelled bot sequences (21 in test set).')
print('Both models limited by data scarcity, not architecture.')
print('More labelled bot data would significantly improve results.')

In [ ]:
# Save both models to Google Drive
import os
save_dir = '/content/drive/MyDrive/auction-bot-detector/models'
os.makedirs(save_dir, exist_ok=True)

# Save Transformer
torch.save({
    'model_state_dict': transformer_model.state_dict(),
    'model_type': 'transformer',
    'threshold': 0.4,
    'input_dim': 8,
    'features': ['time', 'time_diff', 'device', 'country',
                 'auction', 'merchandise',
                 'same_auction_flag', 'same_device_flag']
}, f'{save_dir}/transformer_model.pth')

# Save Autoencoder
torch.save({
    'model_state_dict': ae_model.state_dict(),
    'model_type': 'autoencoder',
    'threshold': 0.7,
    'input_dim': 8,
    'features': ['time', 'time_diff', 'device', 'country',
                 'auction', 'merchandise',
                 'same_auction_flag', 'same_device_flag']
}, f'{save_dir}/autoencoder_model.pth')

print('Both models saved successfully.')
print(f'Saved to: {save_dir}')